In [1]:
import json

files = ["followup_data.jsonl", "goals-honesty-data-fr.jsonl"]

for path in files:
    # read every line; each line is one JSON record (one row)
    with open(path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]

    print(f"File: {path}")
    print(f"  Total rows: {len(rows)}")
    print(f"  Column names: {list(rows[0].keys())}")
    print()

File: followup_data.jsonl
  Total rows: 9932
  Column names: ['prompt_text', 'response_text', 'mix_key']

File: goals-honesty-data-fr.jsonl
  Total rows: 10000
  Column names: ['prompt_text', 'response_text', 'mix_key']



In [2]:
import json
import random

random.seed(42)  # fixed seed so the sample is reproducible

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# 1. load both files
followup = load_jsonl("followup_data.jsonl")
goals    = load_jsonl("goals-honesty-data-fr.jsonl")

# 2. shuffle each, then take 5000 from each
random.shuffle(followup)
random.shuffle(goals)
followup_5k = followup[:5000]
goals_5k    = goals[:5000]

# 3. merge and shuffle the combined set
merged = followup_5k + goals_5k
random.shuffle(merged)

# 4. write the merged file
with open("honesty_sft_mixture_10k.jsonl", "w", encoding="utf-8") as f:
    for row in merged:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# 5. confirm
print("followup taken:", len(followup_5k))
print("goals taken:   ", len(goals_5k))
print("merged total:  ", len(merged))
print("columns:       ", list(merged[0].keys()))

followup taken: 5000
goals taken:    5000
merged total:   10000
columns:        ['prompt_text', 'response_text', 'mix_key']


In [1]:
import json
import random

random.seed(42)  # SAME seed as the SFT run — reproduces the exact same shuffle

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# 1. load both files (same as SFT)
followup = load_jsonl("followup_data.jsonl")
goals    = load_jsonl("goals-honesty-data-fr.jsonl")

# 2. shuffle each with the SAME seed -> identical order to the SFT run
random.shuffle(followup)
random.shuffle(goals)

# 3. SFT used [:5000] from each. The REMAINING (unused) rows are [5000:].
followup_rest = followup[5000:]
goals_rest    = goals[5000:]

# 4. merge and shuffle the remaining set
merged_rest = followup_rest + goals_rest
random.shuffle(merged_rest)

# 5. write the remaining set for OPD
with open("honesty_opd_remaining.jsonl", "w", encoding="utf-8") as f:
    for row in merged_rest:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# 6. confirm + verify no overlap with the SFT set
followup_sft = followup[:5000]
goals_sft    = goals[:5000]
print("followup remaining:", len(followup_rest))
print("goals remaining:   ", len(goals_rest))
print("merged remaining:  ", len(merged_rest))

# safety check: confirm zero overlap between SFT and OPD sets
sft_ids  = {json.dumps(r, sort_keys=True) for r in (followup_sft + goals_sft)}
opd_ids  = {json.dumps(r, sort_keys=True) for r in merged_rest}
overlap  = sft_ids & opd_ids
print("overlap between SFT and OPD sets:", len(overlap), "(should be 0)")

followup remaining: 4932
goals remaining:    5000
merged remaining:   9932
overlap between SFT and OPD sets: 0 (should be 0)
